# Chapter 4: Full Robot Simulation

Welcome to the fourth chapter of our hands-on course! In this tutorial, you will focus on creating a complete robot control plan in pycram and gaining an understanding of fundamental concepts in robotic simulation.

## Learning Objectives
By the end of this chapter, you should be able to:
- Load and initialize a robot and environment for simulation.
- Plan and execute a robotic task that includes object detection, grasping, transporting, and placing.
- Understand key concepts like coordinate transformations, pose calculations, and error handling.

Let's get started!


## Step 1: Getting Started

In this step, we’ll set up the environment and load all necessary libraries. We will also initialize the simulation, creating a robot in a kitchen setting.

*Objective:* By the end of this step, you should have a fully initialized simulation environment and robot to work with.


In [ ]:
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.ros.viz_marker_publisher import VizMarkerPublisher
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode, TorsoState
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color
from pycram.external_interfaces.rosprolog_interface import Prolog as knowrob_client
import pycram.ros.joint_state_subscriber as joint_state


extension = ObjectDescription.get_file_extension()

world = BulletWorld(WorldMode.DIRECT)
world.allow_publish_debug_poses = True
viz = VizMarkerPublisher(interval=0.25)
tf = TFBroadcaster()

robot_name = "pr2"
robot = Object(robot_name, ObjectType.ROBOT, f"{robot_name}{extension}", pose=Pose([1, 2, 0]))

apartment = Object("apartment", ObjectType.ENVIRONMENT, f"apartment-small{extension}")
milk = Object("milk", ObjectType.MILK, "milk.stl", pose=Pose([0.5, 2.5, 1], [0, 0, 0, 1]))
milk.color = Color(0, 0, 1, 1)
milk_desig = BelieveObject(names=["milk"])
robot_desig = BelieveObject(names=[robot_name])
apartment_desig = BelieveObject(names=["apartment"])
joint_state_subscriber = joint_state.JointAngleReader()
knowrob = knowrob_client()

print("Ready for the next cell.")

## Step 2: Detecting the Milk

 Let's start where we left off with detecting the milk in the open fridge!


In [ ]:
with simulated_robot:
    start_pose = Pose([1.3, 2.7, 0], [0, 0, 1, 0])
    milk_target_pose = Pose([5.34, 3.55, 0.8])

    NavigateAction([start_pose]).resolve().perform()
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()

    # get door handle link
    query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_handle(Door,Handle), has_urdf_name(Handle, HandleLinkName), has_urdf_name(Door, DoorLinkName)."
    knowrob_result = knowrob.once(query)
    handle_link_name = knowrob_result.get("HandleLinkName")
    
    handle_designator = ObjectPart(names=[handle_link_name], part_of=apartment_desig.resolve())
    closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
    # check if we need to perform an opening or close action, depending on the angle of the door. 0 = Door is closed, 1.5 = Door is open.
    query = "instance_of(Refrigerator, soma:'Refrigerator'), which_door(Refrigerator,Door), which_action(Door, 0, Action)." 
    knowrob_result = knowrob.once(query)
    

## Step 2: Failure Handling

In robotics, ensuring robust behavior is crucial, especially when dealing with unpredictable environments. Failure handling mechanisms allow a robot to respond to unexpected situations and retry operations when they fail, enhancing reliability.

In PyCRAM, failure handling is implemented currently through the `Retry` class, which can be configured to automatically retry a block of code if it encounters an error. This mechanism is particularly useful for actions where temporary failures may occur, but success is possible upon retrying.

## `Retry` Class: `max_tries`

One of the key parameters in PyCRAM’s `Retry` class is `max_tries`. This parameter determines the maximum number of times a block of code will be retried if it fails. This is a very minimalistic approach to failure handling, but it can be extended to include more sophisticated mechanisms. For now, we’ll focus on the basics.

### **Usage Example**

```python
from pycram.failure_handling import Retry
import random

# Define an operation that might fail
def fragile_operation():
    print("Attempting operation...")
    if random.random() < 0.7:  # Simulating a failure condition
        raise Exception("Operation failed")

# Set up the retry mechanism
retry_handler = Retry(max_tries=3)

# Execute with retries
try:
    retry_handler.run(fragile_operation)
except Exception as e:
    print(f"Operation ultimately failed after retries: {e}")
```
Now let's integrate this mechanism into our plan to detect the milk. There are a few things that could go wrong during this operation, such as the robot not being able to open the fridge door or the milk not being detected. 
Exercise: Implement a retry mechanism to handle potential failures during the door opening and milk detection process. 
You can just use the right arm to force the DoorOpeningAction to fail.  
 

In [ ]:
 if knowrob_result.get("Action") == 'OpenAction':
     #Add Failure Handling here
        OpenAction(object_designator_description=handle_designator, arms=[Arms.LEFT],
                    start_goal_location=[closed_location, opened_location]).resolve().perform()
                    
        LookAtAction(targets=[milk_desig.resolve().pose]).resolve().perform()
        object_designator = DetectAction(milk_desig).resolve().perform()
        print(object_designator)

<details>

<summary>Solution</summary>

```python
knowrob.all_solutions('member(X,[1,2,3]).')
```
</details>

## Interlude: Task Outline
Now, the remaining tasks are: picking up the milk, transporting it, and placing it at a specified location, for example a table. While this might seem simple to us, a robot control program must account for many complex details. 

For a robot to effectively pick up the milk, it needs to understand how to position and move its arm precisely, apply the right amount of force, and assess whether the milk is full, half-full, or empty, as each condition affects the grip strength and energy required. Additionally, it must know if the milk is open or closed, which influences how it should be handled and placed to prevent spills.

In this tutorial, we’ll focus on key concepts rather than delving into physical details. The pycram simulation environment, called Bullet World, is designed for rapid testing, allowing the robot to move faster than it would in a real-world setting. To save time, we also teleport the robot instead of having it navigate full paths. This has the advantage that we can quickly write and test plans for the robot.

Here’s what we will cover:

1. Parking both arms for initial positioning.
2. Determining the appropriate grasp based on the object type.
3. Finding a reachable pose for the robot to pick up the object with the chosen arm.
4. Navigating to the pick-up pose, performing the pick-up, and parking arms afterward.
5. Resolving a reachable location for placing the object.
6. Navigating to the placement location, performing the placement, and parking arms post-transport.

### **Question:** Why do you think it's important for the robot to calculate the approach direction when grasping an object?

`Your answer here`

<details>

<summary>Click here for the Solution.</summary>

For many objects, it is essential to calculate which object side is facing the robot, to make grasping as easy as possible. For example we just hard-coded that the robot should always pick up the milk from the front, and the milk is rotated by 180° around the z axis relative to the robot, picking it up from the "Front" would  mean unnaturally reaching around the object to pick it up, since the milks backside is facing the robot. If we just ignored the rotation of the milk completely we would risk ending up with a very instable grasp, potentially damaging or dropping the milk, since the milk is not perfectly round, but has a rectangular shape. Thus, our cleanest option is to calculate the relative orientation of the object to the robot and choose the side that is facing the robot.

</details>

## Step 3: Calculating Reachable Poses for Grasping

Now that we have detected the milk, we need to determine reachable poses for grasping it.

*Objective:* By the end of this step, you should be able to calculate a pose for the robot that allows it to reach and grasp the object.


### The Grasp
Grasping is a challenging aspect of any robot control program. Depending on where the perception system detects the object, its orientation can vary significantly. Ideally, we want to instruct the robot to "pick up the milk from the front" or "from the top" in a way that’s easy for humans to specify. Some objects may be asymmetrical, which means we always need to grasp them from a specific direction. But for other objects, like for example milk, we can grasp them from any side, so the easiest way is to grasp them from the side that is facing the robot.

To address this, we developed the `calculate_object_faces` function. This function takes an object as input and, using rotation matrices and the robot's position, calculates the correct direction to approach the object from, based on its orientation relative to the robot.

<details>

<summary>Click here for the code behind calculate_object_faces</summary>

```Python
def calculate_object_faces(obj_desig: ObjectDesignatorDescription.Object):
    """
    Calculates the faces of an object relative to the robot based on orientation and position.

    This method determines the faces of the object that are directed towards the robot, by calculating vectors from
    the object to the robot's base and using the object's orientation to determine the side and top/bottom faces.

    For side_face, only the x and y components are considered.
    For top_bottom_face, only the z component is considered.

    Args:
        object (ObjectDesignatorDescription.Object): The object whose faces are to be calculated, with an accessible pose attribute.

    Returns:
        list: A list containing two Grasp Enums, where the first element is the face of the object facing the robot,
              and the second element is the top or bottom face of the object.
    """
    oTm = obj_desig.pose
    base_link = RobotDescription.current_robot_description.base_link
    base_link_pose = obj_desig.world_object.world.robot.get_link_pose(base_link)

    object_position = oTm.position_as_list()
    robot_position = base_link_pose.position_as_list()
    vector_to_robot_world = [robot_position[i] - object_position[i] for i in range(3)]

    orientation = oTm.orientation_as_list()
    rotation_matrix = R.from_quat(orientation).as_matrix()
    o_R_w = rotation_matrix.T

    vector_to_robot_local = o_R_w.dot(vector_to_robot_world)

    vector_x, vector_y, vector_z = vector_to_robot_local

    vector_facing = np.array([vector_x, vector_y, 0])
    side_face = calculate_vector_face(vector_facing)

    vector_z = np.array([0, 0, vector_z])
    top_bottom_face = calculate_vector_face(vector_z)

    return [side_face, top_bottom_face]

```
</details>

We’ll have identified the correct grasp based on which side of the object faces the robot. Next, we’ll use `CostmapLocation` to determine a suitable position where the robot can stand to pick up the milk.

### Exercise: Finding a Suitable Pose with "CostmapLocation"

In this exercise, you’ll use `CostmapLocation` to identify a reachable location where the robot can pick up an object (in this case, the milk) using the correct grasp. Here’s a breakdown of what each parameter in `CostmapLocation` represents:

- **`target`**: The designator for the object you want the robot to pick up (e.g., `self.object_designator`).
- **`reachable_for`**: The robot or component that needs access to the target (use `robot_desig.resolve()` for the robot).
- **`reachable_arm`**: The specific arm that will perform the grasp (e.g., `Arms.LEFT`).
- **`used_grasps`**: The list of grasps that are suitable for the target object (e.g., `[Grasp.FRONT]`).
- **`object_in_hand`**: The object the robot is currently holding (only applicable when placing).

#### Code Template

Complete the following code to find a suitable pose with `CostmapLocation`, and add failure handling in case that the CostmapLocation cannot find a suitable pose.

```python
# Define the location using CostmapLocation
pickup_loc = CostmapLocation(
    target=____,  # Use the object designator
    reachable_for=____,  # Resolve the robot designator
    reachable_arm=____,  # Specify the arm you are using
    used_grasps=____,  # Use the determined grasp in a list
    object_in_hand=None  # This is not needed for picking up
).resolve()
```

In [ ]:
#ParkArms
#MoveTorsoAction
#grasp, _ = _
#target_object = perceivedobj

with simulated_robot:
    pickup_loc = CostmapLocation(target=____,
    reachable_for=____,
    reachable_arm=____,
    used_grasps=____,
    object_in_hand=None).resolve()  
    print("Found a reachable location for the pickup location")
    

<details>

<summary>Click here for the Solution.</summary>

```Python
with simulated_robot:
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()

    side_grasp, _ = calculate_object_faces(object_designator)
    pickup_arm = Arms.RIGHT

    try:
        pickup_loc = CostmapLocation(
            target=object_designator,
            reachable_for=robot_desig.resolve(),
            reachable_arm=pickup_arm,
            used_grasps=[side_grasp]).resolve()
    except StopIteration:
        raise ReachabilityFailure(
            f"No reachable location found for the pickup location: {object_designator.pose}"
        )
    print("Found a reachable location for the pickup location")
```

## Step 4: Navigating to the Pickup Location and Performing the Pickup
Once a reachable pose is found, navigate to it and perform a PickUpAction. This action requires the object designator, the selected arm in a list, and the previously determined grasp in a list as parameters.

The pickup action is used as follows:
    
```Python
PickUpAction(object_designator, [pickup_arm], [grasp]).resolve().perform()
```

Note: Don't forget to put your code in a "with simulated_robot" block to ensure the simulation runs correctly.

### Exercise: Write the Plan to Navigate and Pick Up the Milk 

In [ ]:
with simulated_robot:
    # Add your code here   
    print("Picked up the milk!")

<details>

<summary>Click here for the Solution.</summary>

```python
with simulated_robot:
    NavigateActionPerformable(pickup_loc.pose).perform()
    PickUpAction(object_designator, [pickup_arm], [side_grasp]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()
```
</details>


## Step 5: Transport the Object and Final Placement

In this step, we’ll use the calculated pose to pick up the milk and place it at the target location.

*Objective:* By the end of this step, you will have executed a full pick-and-place operation.

Steps to complete:
1. **Close the fridge**: Use `CloseAction` to close the fridge door.
2. **Find place locaiton**: Use `CostmapLocation` to find a suitable pose for the robot to place the milk at the target location. Note: make sure to specify the object thats being carried using the parameter `object_in_hand=object_designator`
3. **Place the milk**: Use the `PlaceAction` to place the milk at the target location.

The place action is used as follows:
```Python
PlaceAction(object_designator, [target_pose], [pickup_arm])
```

### Exercise: Write the plan
Your task now is to write a complete plan that includes transporting the object and closing the fridge. Don't forget the failure handling. The target pose for the milk is:

```python
milk_target_pose = Pose([5.34, 3.55, 0.8])
```

In [ ]:
with simulated_robot:
    # Add your code here   
    print("Placed the milk!")

<details>

<summary>Click here for the Solution.</summary>

```python
with simulated_robot:
    CloseAction(object_designator_description=handle_designator, arms=[opened_location.arms[0]],
                start_goal_location=[opened_location, closed_location]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()

    try:
        place_loc = CostmapLocation(
            target=milk_target_pose,
            reachable_for=robot_desig.resolve(),
            reachable_arm=pickup_arm,
            used_grasps=[side_grasp],
            object_in_hand=object_designator
        ).resolve()
    except StopIteration:
        raise ReachabilityFailure(
            f"No reachable location found for the target location: {milk_target_pose}"
        )

    NavigateActionPerformable(place_loc.pose).perform()
    PlaceAction(object_designator, [milk_target_pose], [pickup_arm]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()
```
</details>


<details>
<summary>Click here if you want to see the full plan.</summary>

```python
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.ros.viz_marker_publisher import VizMarkerPublisher
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode, TorsoState
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color

extension = ObjectDescription.get_file_extension()

world = BulletWorld(WorldMode.DIRECT)
world.allow_publish_debug_poses = True
viz = VizMarkerPublisher()
tf = TFBroadcaster()

robot_name = "pr2"
robot = Object(robot_name, ObjectType.ROBOT, f"{robot_name}{extension}", pose=Pose([1, 2, 0]))

apartment = Object("apartment", ObjectType.ENVIRONMENT, f"apartment-small{extension}")
milk = Object("milk", ObjectType.MILK, "milk.stl", pose=Pose([0.5, 2.5, 1], [0, 0, 0, 1]))
milk.color = Color(0, 0, 1, 1)
milk_desig = BelieveObject(names=["milk"])
robot_desig = BelieveObject(names=[robot_name])
apartment_desig = BelieveObject(names=["apartment"])

with simulated_robot:
    start_pose = Pose([1.3, 2.7, 0], [0, 0, 1, 0])
    milk_target_pose = Pose([5.34, 3.55, 0.8])

    NavigateAction([start_pose]).resolve().perform()
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()
    
    handle_designator = ObjectPart(names=["handle_cab3_door_top"], part_of=apartment_desig.resolve())
    closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
    OpenAction(object_designator_description=handle_designator, arms=[closed_location.arms[0]],
               start_goal_location=[closed_location, opened_location]).resolve().perform()
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    NavigateAction([start_pose]).resolve().perform()
    LookAtAction(targets=[milk_desig.resolve().pose]).resolve().perform()
    object_designator = DetectAction(milk_desig).resolve().perform()

    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()

    side_grasp, _ = calculate_object_faces(object_designator)
    pickup_arm = Arms.LEFT if closed_location.arms[0] == Arms.RIGHT else Arms.RIGHT

    try:
        pickup_loc = CostmapLocation(
            target=object_designator,
            reachable_for=robot_desig.resolve(),
            reachable_arm=pickup_arm,
            used_grasps=[side_grasp]
        ).resolve()
    except StopIteration:
        raise ReachabilityFailure(
            f"No reachable location found for the pickup location: {object_designator.pose}"
        )

    NavigateActionPerformable(pickup_loc.pose).perform()
    PickUpAction(object_designator, [pickup_arm], [side_grasp]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()
    CloseAction(object_designator_description=handle_designator, arms=[opened_location.arms[0]],
                start_goal_location=[opened_location, closed_location]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()

    try:
        place_loc = CostmapLocation(
            target=milk_target_pose,
            reachable_for=robot_desig.resolve(),
            reachable_arm=pickup_arm,
            used_grasps=[side_grasp],
            object_in_hand=object_designator
        ).resolve()
    except StopIteration:
        raise ReachabilityFailure(
            f"No reachable location found for the target location: {milk_target_pose}"
        )

    NavigateActionPerformable(place_loc.pose).perform()
    PlaceAction(object_designator, [milk_target_pose], [pickup_arm]).resolve().perform()
    ParkArmsActionPerformable(Arms.BOTH).perform()
```
</details>

Now we have our first complete transport plan! While this is a solid foundation for understanding the basics, it lacks some advanced cognitive capabilities.


## Reflection Questions
1. Why is it important to handle retries in robotic actions?
2. What potential errors might occur when resolving designators?
3. How can the retry mechanism be improved to handle different types of failures?
4. 
## What’s Missing?

- **Failure Handling**: Adding failure handling is crucial. It not only makes the robot more resilient but also highlights the complexities a robot control system must navigate. Designing a generalized failure-handling approach requires the control program to be semantically transparent, allowing it to retry, replan, transform, or adapt to different types of failures without hidden information.

- **Learning from Experience**: Another concept is memory-based learning. By logging each action the robot performs, we can replay and analyze this data to help the robot identify and correct mistakes. You’ll find more on this in Chapter 6’s extra materials.

- **Enhanced Physical Understanding**: In this simulation, the robot's physical interactions are simplified, which bypasses some real-world challenges. Implementing true physical understanding would involve additional considerations beyond what’s covered here.
